# B固定检出与定位拒读：Card94用户运行包

Run all执行已采纳固定2校准＋2确认新来源，三臂OFF/P0/PM、六裁剪、双钥、四实际E41、全137状态。
先保存两校准宿主盲结果，再监督计算tau/lambda并持久冻结；确认任何sourceprep均在冻结之后。
检出与候选集定位分开；集合歧义只允许全候选一致支持的组位读出。OFF无目标、错钥归属、缺失组假读分别保留。

完整成功路径：生成pipeline4、DiT800、native400、局部FFT梯度100；VAEload4、D12/E288、接收FFT288、MP4编码/探测/解码各12。
writer FFT前向100＋autograd100另计。固定手选名单只支持经验结果，不是总体FPR或认证位置保证；全拒读不是成功。
每PM正确host几何存在80／缺失112，名义支持72／存在未观测8，净消息仍32。

首代码单元仅mount Drive。已有attempt/执行残留时后续全部只读，跳过源码/依赖/模型，不续算、不补失败。
新attempt直接用本地可用源码或GitHub现有dev分支，无手工ZIP上传、源码内嵌或摘要/版本/Git状态准入。
当前包仅CPU tiny/fake与静态检查；完整Colab运行及真实tau/lambda尚未执行。发布后才可从远端取得本候选。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, subprocess
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Fixed-Detection-Confidence-V1/fixed-2cal-2confirm-v1')
def execution_artifacts(output):
    output = Path(output)
    names = ('attempt.json', 'result.json', 'checkpoint.json', 'ledger.json', 'thresholds.json', 'evaluation.json', 'protocol.json')
    found = [n+s for n in names for s in ('', '.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for folder in ('hosts', 'receipts', 'markers', 'blind', 'private', 'decisions'):
        root = output/folder
        if root.is_dir():
            found.extend(str(p.relative_to(output)) for p in root.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    SOURCE_ROOT=Path('/content/SC-SSTW-b-fixed-detection-confidence-v1')
    if not SOURCE_ROOT.exists():
        subprocess.run(['git','clone','--depth','1','--single-branch','--branch',
            'dev/local-joint-state-payload-v1','https://github.com/RICHAAARC/SC-SSTW.git',str(SOURCE_ROOT)],check=True)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/b_fixed_detection_confidence_v1_run.py').is_file():
        raise FileNotFoundError('Card94 runner unavailable in actual source directory: '+str(SOURCE_ROOT))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Available source',SOURCE_ROOT)
else:
    print('Existing attempt: saved records only; no source/setup/model reentry')


一次固定执行：只修复实际缺失导入或媒体工具；进程异常／中断不重启，所有已尝试槽保留。

In [ ]:

import subprocess, sys, time
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))
    with (OUTPUT/'runner.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'experiments.wan_state_clock.b_fixed_detection_confidence_v1_run', '--output', str(OUTPUT)],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            time.sleep(30)
            try:
                progress = json.loads((OUTPUT/'progress.json').read_text())
                print('Units', progress['returned'], '/', progress['planned'], 'active', progress['active'], flush=True)
            except (OSError,ValueError,KeyError,TypeError) as error:
                print('Progress read unavailable; observing same worker:',type(error).__name__,str(error),flush=True)
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill(); process.wait()
        print('Stopped; saved attempts consumed. Later Run all is read-only.')
        raise
    print('Runner return code',process.returncode,'; inspect saved results and runner.log. No retry.')
else:
    print('Existing attempt: source/setup/physical execution skipped.')


读取已保存结果；逐帧／相邻帧残差、源身份、消息、错误归因和擦除均在原文件中保留。

In [ ]:

for name in ('result.json','thresholds.json','evaluation.json'):
    path = OUTPUT/name
    if not path.is_file():
        print('Not available:',name); continue
    try: value=json.loads(path.read_text())
    except (OSError,ValueError) as error:
        print('Read failure',name,type(error).__name__,str(error)); continue
    if name=='result.json':
        print('Execution',value.get('status'),'threshold state',value.get('thresholds_status'),'counts',value.get('counts'))
    elif name=='thresholds.json':
        print('Calibration',value.get('status'),'tau',value.get('tau'),'lambda',value.get('lambda'))
    else:
        print('Confirmation',value.get('confirmation'))
        for host,row in value.get('hosts',{}).items():
            print(host,row['role'],'null detections',row['null_detections'],'wrong singleton',row['positive_singleton_wrong_origin'],
                  'physical present/missing',row['positive_geometry_present'],row['positive_geometry_missing'],
                  'nominal/unobserved',row['positive_nominal_supported'],row['positive_present_unobserved'],
                  'readable',row['positive_readable_present'],'utility',row['utility_status'])
ledger={}
try:
    if (OUTPUT/'ledger.json').is_file(): ledger=json.loads((OUTPUT/'ledger.json').read_text())
except (OSError,ValueError) as error: print('Ledger snapshot read failure',type(error).__name__,str(error))
for path in (OUTPUT/'receipts').rglob('*.json') if (OUTPUT/'receipts').is_dir() else ():
    try:
        row=json.loads(path.read_text()); ledger[row['slot']]=row
    except (OSError,ValueError,KeyError) as error: print('Receipt read failure',path,type(error).__name__,str(error))
unfinished=[(slot,row.get('status'),row.get('error'),row.get('unavailable_from')) for slot,row in ledger.items() if row.get('status')!='RETURNED']
print('Retained unreturned slots',len(unfinished),'/',len(ledger),'first20',unfinished[:20])
print('Fixed hand-chosen sources: empirical only; no population FPR, certified location or scientific PASS.')
print('Full failures: ledger.json / receipts. P0-PM same true-phase raw BER, coverage and all erasures: evaluation.json. Output',OUTPUT)
